# 🌌 MURE-AGI Auto-Training Pipeline
Run all cells sequentially.

In [ ]:
# 1. Environment Setup
import torch
major_version, minor_version = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0,0)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
if major_version >= 8:
    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes vllm
else:
    !pip install --no-deps "xformers<0.0.27" "trl==0.8.6" peft accelerate bitsandbytes vllm
print('✅ Dependencies installed!')


In [ ]:
# 2. File Systems
from google.colab import drive
import os, torch, gc, json, random
drive.mount('/content/drive')
ROOT_DIR = '/content/drive/MyDrive/mure_auto_train'
DATA_DIR = os.path.join(ROOT_DIR, 'dataset')
CHECKPOINT_DIR = os.path.join(ROOT_DIR, 'checkpoints')
for p in [DATA_DIR, CHECKPOINT_DIR]: os.makedirs(p, exist_ok=True)
DATASET_PATH = os.path.join(DATA_DIR, 'mure_1m_dataset.jsonl')


In [ ]:
# 3. Dataset Generation (1M Rules) using vLLM
from tqdm.auto import tqdm
TARGET = 1_000_000
BATCH_SIZE = 2500
HAS_VRAM = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 12e9

def get_count():
    if not os.path.exists(DATASET_PATH): return 0
    with open(DATASET_PATH, "rb") as f: return sum(1 for _ in f)

current_count = get_count()
if current_count < TARGET:
    print(f"Generating Dataset... Current: {current_count}/{TARGET}")
    if HAS_VRAM:
        from vllm import LLM, SamplingParams
        llm = LLM(model="Qwen/Qwen2.5-1.5B-Instruct", max_model_len=512, enforce_eager=True, gpu_memory_utilization=0.5)
        params = SamplingParams(temperature=0.8, top_p=0.9, max_tokens=100)
        seeds = ["quantum computing", "neural pathway", "economic shift", "biological mutation", "social engineering", "climate change"]
        pbar = tqdm(total=TARGET, initial=current_count, desc="⚡ Generating Logic")
        
        while current_count < TARGET:
            prompts = [f"Describe a complex causal chain for {random.choice(seeds)}. Cause -> Effect:" for _ in range(BATCH_SIZE)]
            outputs = llm.generate(prompts, params, use_tqdm=False)
            
            with open(DATASET_PATH, "a") as f:
                for out in outputs:
                    text = out.outputs[0].text.strip()
                    f.write(json.dumps({"instruction": "Analyze cause and effect", "input": "", "output": text}) + "\n")
                f.flush(); os.fsync(f.fileno())
            current_count += BATCH_SIZE
            pbar.update(BATCH_SIZE)
            
        del llm
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        print("🧹 vLLM memory cleared.")
    else:
        print("⚠️ GPU memory low or missing. Falling back to synthetic logic replication.")
        # Synthetic fast fallback if VRAM is low
        with open(DATASET_PATH, "a") as f:
            for i in tqdm(range(TARGET - current_count)):
                f.write(json.dumps({"instruction": "Analyze cause and effect", "input": "", "output": f"If factor {random.randint(1, 100)} changes, then outcome {random.randint(1, 100)} occurs."}) + "\n")
else:
    print("✅ 1M Rule Dataset Ready.")


In [ ]:
# 4. Unsloth QLoRA (4-bit) Finetuning
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-3B-Instruct',
    max_seq_length=1024, load_in_4bit=True, dtype=None
)
model = FastLanguageModel.get_peft_model(
    model, r=16, 
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], 
    lora_alpha=16, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42
)

dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
def format_prompts(e):
    return {'text': f"<|im_start|>user\n{e.get('instruction', '')}<|im_end|>\n<|im_start|>assistant\n{e.get('output', '')}<|im_end|>"}
dataset = dataset.map(format_prompts, batched=False)

trainer = SFTTrainer(
    model=model, 
    tokenizer=tokenizer, 
    train_dataset=dataset, 
    dataset_text_field='text', 
    max_seq_length=1024, 
    args=TrainingArguments(
        per_device_train_batch_size=4, 
        gradient_accumulation_steps=4, 
        max_steps=1000, 
        learning_rate=2e-4, 
        fp16=not torch.cuda.is_bf16_supported(), 
        bf16=torch.cuda.is_bf16_supported(), 
        output_dir=CHECKPOINT_DIR,
        optim="adamw_8bit"
    )
)

print('🚀 Train start!')
trainer.train()
model.save_pretrained(os.path.join(ROOT_DIR, 'MURE_Final_LoRA'))
tokenizer.save_pretrained(os.path.join(ROOT_DIR, 'MURE_Final_LoRA'))
print('✅ Finished.')
